# Notebook 02 — Simulação de Serving Distribuído

## Objetivos

1. Simular **tensor parallelism** para multiplicação matricial (Megatron-LM, Shoeybi et al., 2019)
2. Simular **pipeline parallelism** com micro-batching e analisar pipeline bubbles (GPipe, Huang et al., 2019)
3. Implementar **continuous batching** e comparar com static batching (Orca, Yu et al., 2022)
4. Gerenciar **KV-Cache com PagedAttention** durante serving (Kwon et al., 2023)
5. Explorar **speculative decoding** para redução de latência (Leviathan et al., 2023)

## Contexto Teórico

> *"Para camadas totalmente conectadas, que dominam a computação em Transformers, duas
> estratégias de particionamento emergem: particionamento por colunas e particionamento
> por linhas."* — Seção 'Tensor Parallelism para LLMs'

> *"Continuous batching trata cada iteração de geração como unidade de scheduling
> independente. Quando uma requisição completa, ela é imediatamente removida do batch
> e uma nova pode ser inserida."* — Seção 'Continuous Batching e Scheduling Dinâmico'

**Referências:**
- Shoeybi et al. (2019). Megatron-LM. arXiv:1909.08053.
- Huang et al. (2019). GPipe. NeurIPS.
- Yu et al. (2022). Orca. OSDI.
- Kwon et al. (2023). PagedAttention. SOSP.
- Leviathan et al. (2023). Speculative Decoding. ICML.

In [ ]:
# Imports
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

sys.path.insert(0, os.path.abspath('..'))

from src.data_preprocessing import WorkloadPreprocessor
from src.model import (
    TensorParallelismSimulator,
    PipelineParallelismSimulator,
    ContinuousBatchingScheduler,
    StaticBatchingScheduler,
    ScheduledRequest,
    PagedKVCacheManager,
    SpeculativeDecoder,
)
from src.training import InferenceRunner, ServingConfig
from src.evaluation import ServingMetricsEvaluator
from src.utils import save_metrics, ensure_output_dirs

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

output_dirs = ensure_output_dirs()
evaluator = ServingMetricsEvaluator(output_dir='../outputs/figures')

print('Imports e configuração carregados.')

## 1. Tensor Parallelism — Particionamento de Matrizes

Conforme **Shoeybi et al. (2019)** — Megatron-LM — descrito na seção
*Tensor Parallelism para Large Language Models*:

> *"No particionamento por colunas, uma matriz de pesos $W \in \mathbb{R}^{m \times n}$
> é dividida em $p$ partições: $W = [W_1 | W_2 | ... | W_p]$, onde cada
> $W_i \in \mathbb{R}^{m \times n/p}$."*

Vamos simular esta operação e medir o speedup.

In [ ]:
# Tensor Parallelism: particionamento por colunas
rng = np.random.default_rng(42)
dim = 2048  # Simula hidden_dim
batch_size = 64

X = rng.standard_normal((batch_size, dim)).astype(np.float32)
W = rng.standard_normal((dim, dim)).astype(np.float32)

print(f'Dimensões: X={X.shape}, W={W.shape}')
print(f'Simula multiplicação QKV em self-attention\n')

for n_gpus in [1, 2, 4, 8]:
    sim = TensorParallelismSimulator(num_devices=n_gpus, comm_latency_factor=0.05)
    output, metrics = sim.simulate_forward_column_parallel(X, W)
    print(f'GPUs={n_gpus}: tempo={metrics["total_time"]*1000:.3f}ms, '
          f'speedup={metrics.get("speedup_vs_single", 1.0):.2f}x, '
          f'output shape={output.shape}')

In [ ]:
# Validação: resultado paralelo == resultado sequencial
sim_2gpu = TensorParallelismSimulator(num_devices=2)
output_parallel, _ = sim_2gpu.simulate_forward_column_parallel(X, W)
output_sequential = X @ W

max_error = np.max(np.abs(output_parallel - output_sequential))
print(f'Erro máximo (paralelo vs. sequencial): {max_error:.2e}')
print(f'Resultado idêntico: {np.allclose(output_parallel, output_sequential, atol=1e-5)}')

### 1.1. Benchmark de Tensor Parallelism

Conforme o material da aula:

> *"A eficiência do tensor parallelism depende criticamente da latência de comunicação
> entre dispositivos. Para NVLink (900 GB/s), é eficiente em até 8 GPUs."*

In [ ]:
# Benchmark: speedup vs. tamanho de matriz e número de GPUs
runner = InferenceRunner()
tp_results = runner.run_tensor_parallelism_benchmark(
    matrix_sizes=[256, 512, 1024, 2048, 4096],
    gpu_counts=[1, 2, 4, 8],
    batch_size=64,
)

fig = evaluator.plot_tensor_parallelism_speedup(
    tp_results, save_name='02_tensor_parallelism_speedup.png'
)
plt.show()

# Tabela de resultados
print('\nResultados de Tensor Parallelism:')
pivot = tp_results.pivot_table(
    index='matrix_dim', columns='num_gpus',
    values='speedup', aggfunc='mean'
).round(2)
pivot

## 2. Pipeline Parallelism e Micro-batching

Conforme **Huang et al. (2019)** — GPipe — discutido na seção
*Pipeline Parallelism e Micro-batching*:

> *"Se um batch é dividido em $M$ micro-batches e o pipeline tem $P$ estágios,
> a fração de tempo perdida em bubbles é aproximadamente:"*

$$\text{bubble\_fraction} = \frac{P-1}{M+P-1}$$

> *"Para $M \gg P$, esta fração tende a zero."*

In [ ]:
# Pipeline Parallelism: efeito de micro-batching na pipeline bubble
pp_results = runner.run_pipeline_benchmark(
    stage_counts=[2, 4, 8],
    micro_batch_counts=[1, 2, 4, 8, 16, 32],
    batch_size=64,
    hidden_dim=64,
)

fig = evaluator.plot_bubble_fraction(pp_results, save_name='02_pipeline_bubbles.png')
plt.show()

# Tabela: bubble fraction
print('\nPipeline Bubble Fraction:')
pivot_pp = pp_results.pivot_table(
    index='num_stages', columns='num_micro_batches',
    values='bubble_fraction', aggfunc='mean'
).round(3)
pivot_pp

In [ ]:
# Visualização do schedule do pipeline
sim_pp = PipelineParallelismSimulator(num_stages=4, hidden_dim=32)
batch = rng.standard_normal((32, 32)).astype(np.float32)
_, pp_metrics = sim_pp.simulate_pipeline(batch, num_micro_batches=6)

schedule = pp_metrics['schedule']

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(schedule.T, aspect='auto', cmap='tab20',
               interpolation='nearest')
ax.set_xlabel('Pipeline Step')
ax.set_ylabel('Estágio')
ax.set_title(f'Schedule do Pipeline (P={pp_metrics["num_stages"]}, '
             f'M={pp_metrics["num_micro_batches"]})\n'
             f'Bubble fraction = {pp_metrics["bubble_fraction"]:.3f}, '
             f'Utilização = {pp_metrics["utilization"]:.1%}')
ax.set_yticks(range(pp_metrics['num_stages']))
ax.set_yticklabels([f'Estágio {i}' for i in range(pp_metrics['num_stages'])])
plt.colorbar(im, ax=ax, label='Micro-batch ID (-1 = bubble)')
plt.tight_layout()
plt.savefig('../outputs/figures/02_pipeline_schedule.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Continuous Batching vs. Static Batching

Conforme seção *Continuous Batching e Scheduling Dinâmico* (Yu et al., 2022):

> *"Static batching sofre de ineficiência quando requisições têm comprimentos de saída
> variáveis: requisições curtas devem aguardar as longas, desperdiçando capacidade."*

> *"Em benchmarks, vLLM alcança throughput 2-4× superior ao HF TGI em modelos de 13B,
> e até 24× em cenários com alta variabilidade de comprimento."* — Kwon et al. (2023)

In [ ]:
# Carrega workload
df = WorkloadPreprocessor.load_data('../data/raw/dataset.csv')
df_sample = df.head(200)  # Amostra para simulação rápida

print(f'Workload: {len(df_sample)} requisições')
print(f'Tokens de geração: min={df_sample["actual_gen_tokens"].min()}, '
      f'max={df_sample["actual_gen_tokens"].max()}, '
      f'mean={df_sample["actual_gen_tokens"].mean():.0f}')

In [ ]:
# Continuous Batching
config = ServingConfig(max_batch_size=16, max_total_tokens=4096)
runner_cb = InferenceRunner(config)
result_cb = runner_cb.run_continuous_batching(df_sample)

print('=== Continuous Batching ===')
print(f'Requisições completadas: {result_cb["total_completed"]}')
print(f'Total de steps: {result_cb["total_steps"]}')
print(f'Latência média: {result_cb["avg_latency_steps"]:.1f} steps')
print(f'Latência P95: {result_cb["p95_latency_steps"]:.1f} steps')
print(f'Latência P99: {result_cb["p99_latency_steps"]:.1f} steps')
print(f'Throughput: {result_cb["throughput_req_per_step"]:.4f} req/step')

In [ ]:
# Static Batching (baseline)
runner_sb = InferenceRunner(config)
result_sb = runner_sb.run_static_batching(df_sample)

print('=== Static Batching ===')
print(f'Requisições completadas: {result_sb["total_completed"]}')
print(f'Total de steps: {result_sb["total_steps"]}')
print(f'Latência média: {result_sb["avg_latency_steps"]:.1f} steps')
print(f'Latência P95: {result_sb["p95_latency_steps"]:.1f} steps')
print(f'Latência P99: {result_sb["p99_latency_steps"]:.1f} steps')

In [ ]:
# Comparação visual
fig = evaluator.plot_batching_comparison(
    result_cb['history'],
    result_sb,
    save_name='02_batching_comparison.png',
)
plt.show()

# Comparação numérica
speedup = result_sb['total_steps'] / max(result_cb['total_steps'], 1)
print(f'\n--- Comparação ---')
print(f'Continuous Batching: {result_cb["total_steps"]} steps')
print(f'Static Batching:     {result_sb["total_steps"]} steps')
print(f'Speedup (continuous vs static): {speedup:.2f}x')
print(f'\nConfirma resultado teórico: continuous batching é mais eficiente '
      f'com workloads de comprimento variável (Yu et al., 2022).')

## 4. Speculative Decoding

Conforme **Leviathan et al. (2023)**, seção *Speculative Decoding para Redução de Latência*:

> *"O speedup teórico depende da acceptance rate $\alpha$ do modelo draft:
> $\text{speedup} = \frac{K \cdot \alpha}{\text{cost}_{\text{draft}} + \text{cost}_{\text{verify}}}$"*

> *"Modelos draft bem escolhidos alcançam acceptance rates de 70-85%,
> resultando em speedups de 2-3× sem degradação de qualidade."*

In [ ]:
# Speculative Decoding com diferentes acceptance rates
token_counts = [32, 64, 128, 256, 512]

print('=== Speculative Decoding (α=0.80, K=5) ===')
decoder = SpeculativeDecoder(
    draft_acceptance_rate=0.80,
    draft_speed_factor=0.15,
    speculation_length=5,
    main_model_latency_ms=50.0,
    seed=42,
)

results_spec = decoder.compare_with_autoregressive(token_counts)

for r in results_spec:
    print(f'Tokens={r["total_tokens"]:3d}: '
          f'auto={r["autoregressive_latency_ms"]:.0f}ms, '
          f'spec={r["speculative_latency_ms"]:.0f}ms, '
          f'speedup={r["speedup"]:.2f}x, '
          f'rodadas={r["rounds"]}')

In [ ]:
# Visualização de Speculative Decoding
fig = evaluator.plot_speculative_decoding_analysis(
    results_spec, save_name='02_speculative_decoding.png'
)
plt.show()

In [ ]:
# Efeito da acceptance rate no speedup
alphas = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
fixed_tokens = 256

fig, ax = plt.subplots(figsize=(10, 6))
speedups_by_alpha = []

for alpha in alphas:
    dec = SpeculativeDecoder(
        draft_acceptance_rate=alpha,
        draft_speed_factor=0.15,
        speculation_length=5,
        main_model_latency_ms=50.0,
        seed=42,
    )
    result = dec.simulate_generation(fixed_tokens)
    speedups_by_alpha.append(result['speedup'])

ax.plot(alphas, speedups_by_alpha, 'o-', color='steelblue', linewidth=2, markersize=8)
ax.set_xlabel('Acceptance Rate (α)', fontsize=12)
ax.set_ylabel('Speedup', fontsize=12)
ax.set_title(f'Speculative Decoding: Speedup vs. Acceptance Rate\n'
             f'(K=5, tokens={fixed_tokens}, Leviathan et al., 2023)', fontsize=13)
ax.axhline(1.0, color='red', linestyle='--', alpha=0.5, label='Baseline (1×)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/02_speculative_alpha_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Salvando Métricas

Todas as métricas são persistidas para análise no notebook de avaliação.

In [ ]:
# Salva métricas para o notebook de avaliação
# Remove 'history' e 'schedule' (não serializáveis diretamente)
cb_save = {k: v for k, v in result_cb.items() if k not in ('history', 'kv_cache_stats')}
sb_save = {k: v for k, v in result_sb.items() if k != 'history'}

save_metrics(
    {'continuous_batching': cb_save, 'static_batching': sb_save},
    '../outputs/logs/batching_comparison.json',
    metadata={'workload_size': len(df_sample), 'config': {
        'max_batch_size': config.max_batch_size,
        'max_total_tokens': config.max_total_tokens,
    }},
)

save_metrics(
    {'speculative_decoding': results_spec},
    '../outputs/logs/speculative_decoding.json',
)

print('Métricas salvas em outputs/logs/')

## Resumo

Neste notebook simulamos os principais componentes de serving distribuído:

1. **Tensor Parallelism** — Particionamento de matrizes W em p GPUs (Megatron-LM)
2. **Pipeline Parallelism** — Micro-batching reduz bubble fraction: $(P-1)/(M+P-1)$
3. **Continuous Batching** — Superior ao static batching para workloads variáveis (Orca)
4. **Speculative Decoding** — Speedup de 2-3× com acceptance rate ~80% (Leviathan et al., 2023)

**→ Próximo notebook:** [03_avaliacao.ipynb](03_avaliacao.ipynb) — Avaliação de SLOs,
benchmarks comparativos e análise de trade-offs.